# **Voyage Analytics: Flight Price Prediction with MLOps**


##### **Project Type** - Regression / MLOps  
##### **Contribution** - Individual  
##### **Team Member 1 - Tarun Sreekar Parasa**


# **Project Summary -**

This project develops a production-oriented regression solution for predicting flight prices from the Voyage Analytics travel dataset and prepares the selected model for later deployment through an end-to-end MLOps workflow. The dataset contains 271,888 flight records linked to individual travel journeys. Each record describes the origin, destination, flight type, airline agency, travel time, distance, date and observed flight price. The business objective is to estimate a reasonable flight price from the information available before a booking is completed, while also designing the modeling process in a way that can be reused by Flask, Docker, Kubernetes, Apache Airflow, Jenkins and MLflow components in the wider capstone project.

The analysis begins with data quality checks, structural validation and exploratory data analysis. The raw dataset contains no missing values and no duplicate rows. Dates are converted to a proper datetime format and calendar features such as year, month and day of week are derived. Identifier columns such as `travelCode` and `userCode` are retained for validation and grouping but are excluded from model inputs because identifiers can encourage memorization without providing transferable pricing information. A key finding is that this dataset is highly structured: for a given combination of origin, destination, flight type and agency, the recorded fare is effectively fixed. This property explains why tree-based models can achieve extremely high predictive performance and is discussed as an important limitation when interpreting the results.

To reduce leakage, the data is split by `travelCode` with `GroupShuffleSplit`. Each trip contains paired flight records, so a normal row-level random split could place records from the same journey in both training and testing data. Group-based splitting ensures that all rows belonging to a travel journey remain on only one side of the split. Three regression approaches are compared: Linear Regression as an interpretable baseline, Decision Tree Regression as a non-linear model, and Random Forest Regression as the final ensemble model. Performance is assessed using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE) and R-squared (R²).

The baseline Linear Regression captures broad price patterns but cannot fully represent the non-linear interactions between route, class and agency. Tree-based approaches perform substantially better. In the validated local benchmark, Linear Regression produced an R² of approximately 0.921, while Decision Tree and Random Forest models achieved near-perfect R² because they can learn the deterministic pricing combinations present in this dataset. The final Random Forest model is selected for deployment because it combines strong predictive accuracy with greater stability than a single tree.

The completed notebook saves the entire preprocessing-and-model pipeline with Joblib, allowing exactly the same transformations to be applied during API inference. This design supports the next MLOps stages of the capstone: REST API serving, containerization, Kubernetes deployment, workflow orchestration, CI/CD automation and MLflow experiment tracking.


# **GitHub Link -**

**GitHub Repository:** [https://github.com/Sreekar-DS/Voyage-Analytics-Integrating-MLOps-in-Travel](https://github.com/Sreekar-DS/Voyage-Analytics-Integrating-MLOps-in-Travel)


# **Problem Statement**


The travel company requires a regression model that can predict the price of a flight from route, flight characteristics and booking-related information. The model should be accurate, reproducible and suitable for deployment through a REST API. The modeling workflow must therefore include careful data validation, leakage-aware train/test splitting, feature preprocessing, model comparison, evaluation, model persistence and a clear path to production deployment.


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required. 
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits. 
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule. 

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


### Dataset Loading

In [ ]:
# Load Dataset
# The notebook supports a cloned GitHub repository, a manually uploaded Colab file,
# Google Drive, and the local validation path used while developing this submission.
possible_paths = [
    Path('data/raw/flights.csv'),
    Path('/content/flights.csv'),
    Path('/content/drive/MyDrive/Voyage-Analytics/flights.csv'),
    Path('/mnt/data/voyage_work/flights.csv')
]

DATA_PATH = next((path for path in possible_paths if path.exists()), None)

if DATA_PATH is None:
    try:
        from google.colab import files
        print('flights.csv was not found automatically. Please upload it now.')
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError(
            'Could not locate flights.csv. Place it in data/raw/, upload it to Colab, '
            'or update one of the paths in possible_paths.'
        ) from exc

flight_df = pd.read_csv(DATA_PATH)
print(f'Loaded dataset from: {DATA_PATH}')


### Dataset First View

In [ ]:
# Dataset First Look
display(flight_df.head())
display(flight_df.tail())


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print(f'Rows: {flight_df.shape[0]:,}')
print(f'Columns: {flight_df.shape[1]}')


### Dataset Information

In [ ]:
# Dataset Info
flight_df.info()


#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicate_count = flight_df.duplicated().sum()
print(f'Duplicate rows: {duplicate_count:,}')


#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
missing_summary = flight_df.isna().sum().to_frame('missing_count')
missing_summary['missing_percent'] = (missing_summary['missing_count'] / len(flight_df) * 100).round(2)
display(missing_summary)


In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10, 4))
sns.heatmap(flight_df.isnull(), cbar=False, yticklabels=False)
plt.title('Missing-Value Pattern in Flights Dataset')
plt.show()


### What did you know about your dataset?

The flights dataset contains **271,888 rows and 10 raw columns**. It is a clean transactional dataset with no missing values and no exact duplicate rows. `travelCode` identifies a journey and `userCode` links the flight record to the users table. The prediction target is `price`. The date column requires conversion from text to datetime before feature engineering. The dataset contains paired flight records per journey, so `travelCode` will be used as the grouping variable during train/test splitting to prevent trip-level leakage.


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
flight_df.columns.tolist()


In [ ]:
# Dataset Describe
display(flight_df.describe(include='all').T)


### Variables Description 

- **travelCode:** Unique identifier for a travel journey. Used only for grouped splitting, not as a predictor.  
- **userCode:** User identifier linked to the users dataset. Excluded from the model to avoid user-ID memorization.  
- **from:** Origin city.  
- **to:** Destination city.  
- **flightType:** Travel class/category.  
- **price:** Flight price and regression target.  
- **time:** Flight duration.  
- **distance:** Route distance.  
- **agency:** Flight agency/provider.  
- **date:** Date of travel, later converted to datetime and decomposed into calendar features.


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for column in flight_df.columns:
    print(f'{column}: {flight_df[column].nunique():,} unique values')
    if flight_df[column].nunique() <= 15:
        print(flight_df[column].value_counts(dropna=False).to_dict())
    print('-' * 70)


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
flights = flight_df.copy()
flights['date'] = pd.to_datetime(flights['date'], format='%m/%d/%Y', errors='coerce')

# Calendar features for exploratory analysis and modeling.
flights['year'] = flights['date'].dt.year
flights['month'] = flights['date'].dt.month
flights['dayofweek'] = flights['date'].dt.dayofweek
flights['route'] = flights['from'] + ' → ' + flights['to']
flights['price_per_km'] = flights['price'] / flights['distance'].replace(0, np.nan)

# Structural validation: determine whether fares vary within the same commercial combination.
price_variation = (
    flights.groupby(['from', 'to', 'flightType', 'agency'])['price']
    .nunique()
    .sort_values(ascending=False)
)

print(f'Invalid dates after conversion: {flights["date"].isna().sum()}')
print(f'Number of directed routes: {flights["route"].nunique()}')
print(f'Maximum unique prices within one route/class/agency combination: {price_variation.max()}')
print(f'Date range: {flights["date"].min().date()} to {flights["date"].max().date()}')


### What all manipulations have you done and insights you found?

The main manipulations were: converting `date` to datetime; deriving `year`, `month` and `dayofweek`; creating a human-readable `route`; and creating `price_per_km` for exploratory analysis. The structural validation showed that each exact combination of origin, destination, flight type and agency has a single recorded fare. This is a major modeling insight because it explains the extremely strong performance of non-linear tree models. The result should be interpreted as excellent performance **within the pricing rules represented by this dataset**, not proof that the model can forecast dynamic market prices for unseen airlines, routes or future pricing regimes.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1: Price distribution
plt.figure(figsize=(10, 5))
sns.histplot(flights['price'], bins=40, kde=True)
plt.title('Distribution of Flight Prices')
plt.xlabel('Price')
plt.show()


##### 1. Why did you pick the specific chart?

A histogram with KDE was chosen to show the shape, spread and concentration of the continuous target variable before modeling.


##### 2. What is/are the insight(s) found from the chart?

Flight prices span a wide range and are not concentrated around one narrow value, confirming that the regression target contains meaningful variation.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Understanding the target distribution helps select robust error metrics and prevents relying only on R². MAE and RMSE are therefore reported alongside R².


#### Chart - 2

In [ ]:
# Chart - 2: Flight type frequency
plt.figure(figsize=(8, 5))
sns.countplot(data=flights, x='flightType', order=flights['flightType'].value_counts().index, color='C0')
plt.title('Number of Flights by Flight Type')
plt.show()


##### 1. Why did you pick the specific chart?

A count plot clearly compares the representation of the categorical flight-type classes.


##### 2. What is/are the insight(s) found from the chart?

All flight types are represented with substantial volume, so the model has enough examples to learn class-related pricing patterns.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Balanced operational coverage reduces the risk that the model works only for one travel class.


#### Chart - 3

In [ ]:
# Chart - 3: Average price by flight type
plt.figure(figsize=(8, 5))
sns.barplot(data=flights, x='flightType', y='price', estimator=np.mean, errorbar=None, color='C0')
plt.title('Average Flight Price by Flight Type')
plt.show()


##### 1. Why did you pick the specific chart?

A bar chart directly compares mean prices across travel classes.


##### 2. What is/are the insight(s) found from the chart?

Flight type is a strong pricing factor, with premium classes carrying systematically higher average fares.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business can use class-level price behavior for pricing validation and customer-facing price estimates.


#### Chart - 4

In [ ]:
# Chart - 4: Average price by agency
plt.figure(figsize=(8, 5))
sns.barplot(data=flights, x='agency', y='price', estimator=np.mean, errorbar=None, color='C0')
plt.title('Average Flight Price by Agency')
plt.show()


##### 1. Why did you pick the specific chart?

A bar chart was used because agency is categorical and the business question compares average prices among providers.


##### 2. What is/are the insight(s) found from the chart?

Average fares differ by agency, indicating that the provider contributes predictive information beyond route distance alone.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Agency-specific effects justify including the agency variable in the deployed prediction request.


#### Chart - 5

In [ ]:
# Chart - 5: Most frequent routes
top_routes = flights['route'].value_counts().head(10).sort_values()
plt.figure(figsize=(10, 6))
top_routes.plot(kind='barh')
plt.title('Top 10 Routes by Number of Flight Records')
plt.xlabel('Number of Records')
plt.show()


##### 1. Why did you pick the specific chart?

A horizontal bar chart makes long route names readable while ranking the highest-volume routes.


##### 2. What is/are the insight(s) found from the chart?

Travel demand is not evenly distributed across routes; a smaller group of routes contributes a large share of flight records.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

High-volume routes are important for monitoring because model errors there would affect more predictions.


#### Chart - 6

In [ ]:
# Chart - 6: Most expensive routes
top_route_prices = flights.groupby('route')['price'].mean().nlargest(10).sort_values()
plt.figure(figsize=(10, 6))
top_route_prices.plot(kind='barh')
plt.title('Top 10 Routes by Average Flight Price')
plt.xlabel('Average Price')
plt.show()


##### 1. Why did you pick the specific chart?

A ranked horizontal bar chart highlights the routes with the highest average fare.


##### 2. What is/are the insight(s) found from the chart?

The most expensive routes are generally long-distance connections, reinforcing the importance of route and distance features.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Route-level price differences can support fare benchmarking and anomaly monitoring.


#### Chart - 7

In [ ]:
# Chart - 7: Distance distribution
plt.figure(figsize=(10, 5))
sns.histplot(flights['distance'], bins=35, kde=True)
plt.title('Distribution of Flight Distance')
plt.show()


##### 1. Why did you pick the specific chart?

A histogram was chosen to inspect the coverage and skew of the numerical distance feature.


##### 2. What is/are the insight(s) found from the chart?

The dataset contains both short and long journeys, giving the model broad distance coverage within the nine-city network.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Predictions should still be restricted to supported route patterns because the dataset contains a finite network of cities.


#### Chart - 8

In [ ]:
# Chart - 8: Distance versus price
sample_scatter = flights.sample(min(20000, len(flights)), random_state=RANDOM_STATE)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_scatter, x='distance', y='price', hue='flightType', alpha=0.35)
plt.title('Flight Price versus Distance')
plt.show()


##### 1. Why did you pick the specific chart?

A scatter plot reveals the relationship between two continuous variables while flight type shows segmentation.


##### 2. What is/are the insight(s) found from the chart?

Price increases broadly with distance, but flight type creates distinct price bands and non-linear interactions.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

This confirms that a simple distance-only formula would be inadequate for accurate pricing.


#### Chart - 9

In [ ]:
# Chart - 9: Flight time versus price
plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_scatter, x='time', y='price', hue='agency', alpha=0.35)
plt.title('Flight Price versus Travel Time')
plt.show()


##### 1. Why did you pick the specific chart?

A scatter plot was used to evaluate whether travel time carries information similar to distance and whether agencies form different patterns.


##### 2. What is/are the insight(s) found from the chart?

Travel time and price are positively related, but agency-level patterns remain visible.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Both route characteristics and commercial provider information should be retained for prediction.


#### Chart - 10

In [ ]:
# Chart - 10: Monthly booking volume
monthly_volume = flights.set_index('date').resample('M').size()
plt.figure(figsize=(12, 5))
monthly_volume.plot()
plt.title('Monthly Flight Record Volume')
plt.ylabel('Number of Flight Records')
plt.show()


##### 1. Why did you pick the specific chart?

A time-series line chart is appropriate for identifying changes in travel volume over the dataset period.


##### 2. What is/are the insight(s) found from the chart?

The number of records varies over time, showing that travel activity is not constant across the observation period.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Time-based monitoring will be useful after deployment because future data volume or patterns may drift.


#### Chart - 11

In [ ]:
# Chart - 11: Average price by year
plt.figure(figsize=(8, 5))
sns.barplot(data=flights, x='year', y='price', estimator=np.mean, errorbar=None, color='C0')
plt.title('Average Flight Price by Year')
plt.show()


##### 1. Why did you pick the specific chart?

A yearly bar chart provides a simple check for broad price movement over calendar years.


##### 2. What is/are the insight(s) found from the chart?

Average yearly prices remain structurally stable compared with the much larger differences caused by route, class and agency.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

Calendar variables are retained initially but are expected to be less important than route and service attributes.


#### Chart - 12

In [ ]:
# Chart - 12: Agency and flight-type price heatmap
price_matrix = flights.pivot_table(index='agency', columns='flightType', values='price', aggfunc='mean')
plt.figure(figsize=(8, 5))
sns.heatmap(price_matrix, annot=True, fmt='.1f', cmap='YlGnBu')
plt.title('Average Price by Agency and Flight Type')
plt.show()


##### 1. Why did you pick the specific chart?

A heatmap compactly compares the interaction between two categorical pricing variables.


##### 2. What is/are the insight(s) found from the chart?

Flight class and agency interact to create different average fare levels, supporting the use of non-linear models.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The interaction suggests that models capable of learning combinations can outperform a purely additive baseline.


#### Chart - 13

In [ ]:
# Chart - 13: Price per kilometre by agency
plt.figure(figsize=(9, 5))
sns.boxplot(data=flights, x='agency', y='price_per_km', showfliers=False, color='C0')
plt.title('Price per Kilometre by Agency')
plt.show()


##### 1. Why did you pick the specific chart?

A box plot compares the distribution of normalized price per kilometre across agencies while suppressing extreme visual outliers.


##### 2. What is/are the insight(s) found from the chart?

Price per kilometre differs by agency and route mix, showing that distance alone does not fully determine price.


##### 3. Will the gained insights help creating a positive business impact? 
Are there any insights that lead to negative growth? Justify with specific reason.

The business should avoid using one universal price-per-kilometre rule.


#### Chart - 14 - Correlation Heatmap

In [ ]:
# Chart - 14 - Correlation Heatmap
numeric_cols = ['price', 'time', 'distance', 'year', 'month', 'dayofweek', 'price_per_km']
plt.figure(figsize=(9, 7))
sns.heatmap(flights[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


##### 1. Why did you pick the specific chart?

A correlation heatmap summarizes linear relationships among all engineered numerical variables.


##### 2. What is/are the insight(s) found from the chart?

Distance and time are strongly related, while both are associated with price. Correlation does not capture the important categorical interactions.


#### Chart - 15 - Pair Plot 

In [ ]:
# Chart - 15 - Pair Plot
pair_sample = flights[['price', 'time', 'distance', 'flightType']].sample(3000, random_state=RANDOM_STATE)
sns.pairplot(pair_sample, hue='flightType', corner=True)
plt.show()


##### 1. Why did you pick the specific chart?

A pair plot provides a multivariate view of the main numerical variables and class separation in one figure.


##### 2. What is/are the insight(s) found from the chart?

The pair plot confirms strong route geometry relationships and visible class-based price separation, supporting non-linear modeling.


## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Three business-relevant hypotheses are tested: (1) flight prices differ across flight types; (2) flight prices differ across agencies; and (3) flight distance has a positive monotonic association with price. Because the dataset is very large, statistical significance is interpreted together with practical effect and visualization rather than using the p-value alone.


### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** The flight-price distributions are the same across flight types.  
**H₁:** At least one flight type has a different flight-price distribution.


#### 2. Perform an appropriate statistical test.

In [ ]:
# Kruskal-Wallis test for price differences across flight types
flight_type_groups = [group['price'].values for _, group in flights.groupby('flightType')]
kw_stat_1, p_value_1 = stats.kruskal(*flight_type_groups)
print(f'Kruskal-Wallis statistic: {kw_stat_1:.4f}')
print(f'p-value: {p_value_1:.6g}')
print('Reject H0' if p_value_1 < 0.05 else 'Fail to reject H0')


##### Which statistical test have you done to obtain P-Value?

The **Kruskal-Wallis H-test** was used.


##### Why did you choose the specific statistical test?

It compares the distributions of a continuous outcome across more than two independent groups without requiring the target to be normally distributed.


### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** The flight-price distributions are the same across agencies.  
**H₁:** At least one agency has a different flight-price distribution.


#### 2. Perform an appropriate statistical test.

In [ ]:
# Kruskal-Wallis test for price differences across agencies
agency_groups = [group['price'].values for _, group in flights.groupby('agency')]
kw_stat_2, p_value_2 = stats.kruskal(*agency_groups)
print(f'Kruskal-Wallis statistic: {kw_stat_2:.4f}')
print(f'p-value: {p_value_2:.6g}')
print('Reject H0' if p_value_2 < 0.05 else 'Fail to reject H0')


##### Which statistical test have you done to obtain P-Value?

The **Kruskal-Wallis H-test** was used.


##### Why did you choose the specific statistical test?

Agency contains more than two categories and the price distribution is not assumed to be normal, making a non-parametric multi-group comparison appropriate.


### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**H₀:** Flight distance has no monotonic association with price (Spearman ρ = 0).  
**H₁:** Flight distance has a non-zero monotonic association with price (Spearman ρ ≠ 0).


#### 2. Perform an appropriate statistical test.

In [ ]:
# Spearman rank correlation between distance and price
rho, p_value_3 = stats.spearmanr(flights['distance'], flights['price'])
print(f'Spearman correlation: {rho:.4f}')
print(f'p-value: {p_value_3:.6g}')
print('Reject H0' if p_value_3 < 0.05 else 'Fail to reject H0')


##### Which statistical test have you done to obtain P-Value?

The **Spearman rank-correlation test** was used.


##### Why did you choose the specific statistical test?

Spearman correlation measures monotonic association and does not require a linear relationship or normally distributed variables.


## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values
print(flights.isna().sum())


#### What all missing value imputation techniques have you used and why did you use those techniques?

No missing-value imputation was required because the raw dataset is complete and the date conversion produced no invalid dates. The deployment pipeline still uses explicit preprocessing so future schema validation can detect unexpected missing values rather than silently producing incorrect predictions.


### 2. Handling Outliers

In [ ]:
# Handling Outliers
# Inspect potential extremes, but retain them because they represent valid route/class combinations.
q1, q3 = flights['price'].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
potential_outliers = flights[(flights['price'] < lower_bound) | (flights['price'] > upper_bound)]
print(f'Potential IQR outliers in target: {len(potential_outliers):,}')
print('These rows are retained because high fares are valid observations, not data-entry errors.')


##### What all outlier treatment techniques have you used and why did you use those techniques?

IQR analysis was used for diagnosis only. No target rows were removed because the high prices correspond to genuine long-distance or premium-class combinations. Removing valid expensive flights would distort the business problem and artificially reduce prediction difficulty.


### 3. Categorical Encoding

In [ ]:
# Categorical Encoding
categorical_features = ['from', 'to', 'flightType', 'agency']
numerical_features = ['time', 'distance', 'year', 'month', 'dayofweek']

linear_preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), numerical_features)
])

tree_preprocessor = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features),
    ('num', 'passthrough', numerical_features)
])


#### What all categorical encoding techniques have you used & why did you use those techniques?

One-hot encoding is used for Linear Regression so categorical values are not given an artificial numerical order. Ordinal encoding is used inside the tree pipelines for compact and efficient tree training; the encoder also maps unseen categories to -1 so the API can fail gracefully or flag unsupported values.


### 4. Textual Data Preprocessing 
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


#### 2. Lower Casing

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


#### 3. Removing Punctuations

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


In [ ]:
# Remove White spaces

#### 6. Rephrase Text

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


#### 7. Tokenization

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


#### 8. Text Normalization

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


##### Which text normalization technique have you used and why?

Not applicable. The flight-price regression dataset contains structured tabular features rather than free text, so NLP preprocessing and text vectorization are not required.


#### 9. Part of speech tagging

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


#### 10. Text Vectorization

In [ ]:
# Not applicable: this is a structured tabular regression dataset.


##### Which text vectorization technique have you used and why?

Not applicable. The flight-price regression dataset contains structured tabular features rather than free text, so NLP preprocessing and text vectorization are not required.


### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Feature Manipulation
model_data = flights.copy()

FEATURES = ['from', 'to', 'flightType', 'agency', 'time', 'distance', 'year', 'month', 'dayofweek']
TARGET = 'price'
GROUP_COLUMN = 'travelCode'

X = model_data[FEATURES]
y = model_data[TARGET]
groups = model_data[GROUP_COLUMN]

print('Model features:', FEATURES)
print('Target:', TARGET)


#### 2. Feature Selection

In [ ]:
# Feature Selection
# Identifiers are deliberately excluded from the model.
excluded_columns = ['travelCode', 'userCode', 'date', 'route', 'price_per_km']
print('Excluded from predictors:', excluded_columns)

# Validate the key deterministic pricing structure for transparent interpretation.
max_price_variation = (
    flights.groupby(['from', 'to', 'flightType', 'agency'])['price'].nunique().max()
)
print('Maximum number of unique prices inside one route/class/agency combination:', max_price_variation)


##### What all feature selection methods have you used  and why?

Feature selection was guided by domain logic and leakage prevention. `travelCode` and `userCode` were excluded because they are identifiers. `price_per_km` was excluded because it is calculated using the target price and would create direct target leakage. The raw `date` was replaced by derived calendar features.


##### Which all features you found important and why?

The most important expected predictors are route (`from`, `to`), `flightType`, `agency`, `distance` and `time`. The dataset audit shows that route, class and agency combinations are especially powerful because each such combination has a fixed fare in the supplied data.


### 5. Data Transformation

A target transformation is not required. The final tree-based model does not assume normally distributed residuals, and keeping prices on their original scale makes MAE/RMSE and API predictions directly interpretable to business users.


In [ ]:
# Data Transformation
# No log or power transformation is applied to the target.
# Tree-based models can learn the non-linear pricing structure directly.
print(f'Target skewness: {y.skew():.4f}')


### 6. Data Scaling

In [ ]:
# Data Scaling
# StandardScaler is used only in the linear baseline inside linear_preprocessor.
# Tree-based models use original numerical scales.
print('Scaling strategy defined inside model-specific preprocessing pipelines.')


Standardization is used for the Linear Regression numerical inputs to keep feature magnitudes comparable. Scaling is not required for Decision Tree or Random Forest models because split decisions are invariant to monotonic rescaling of individual features.


### 7. Dimesionality Reduction

Dimensionality reduction is not needed. The final feature set is small, interpretable and operationally meaningful. Applying PCA would reduce transparency and complicate API input validation without a clear performance benefit.


Answer Here.

In [ ]:
# Dimensionality Reduction
# Not applied because the feature space is already compact.
print('Dimensionality reduction not applied.')


No dimensionality-reduction technique was used because the selected tabular features are few and should remain directly interpretable.


Answer Here.

### 8. Data Splitting

In [ ]:
# Data Splitting - leakage-aware grouped split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print('Travel-code overlap:', len(set(groups_train) & set(groups_test)))


##### What data splitting ratio have you used and why? 

An 80:20 split was used. More importantly, the split is grouped by `travelCode`, ensuring that paired records from the same journey do not appear in both training and test sets. This is more appropriate than a simple random row split for this dataset structure.


### 9. Handling Imbalanced Dataset

Class imbalance is not applicable because this is a regression problem with a continuous target rather than discrete target classes.


Answer Here.

In [ ]:
# Handling Imbalanced Dataset
print('Not applicable to continuous-target regression.')


No resampling technique is required for the regression target. Model performance is evaluated directly on the naturally occurring price distribution.


Answer Here.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1: Linear Regression
linear_model = Pipeline([
    ('preprocessor', linear_preprocessor),
    ('model', LinearRegression())
])
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

def regression_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'R2': r2_score(y_true, y_pred)
    }

linear_results = regression_metrics(y_test, linear_pred, 'Linear Regression')
display(pd.DataFrame([linear_results]))


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

Linear Regression is used as an interpretable baseline. It captures the broad additive effects of route, class, agency and numerical travel variables. In the validated benchmark it achieved approximately **MAE 80.63, RMSE 102.10 and R² 0.921**. This is a strong baseline, but it cannot fully represent the non-linear interactions that define the supplied pricing table.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Cross-validation for Linear Regression using travelCode groups
cv = GroupKFold(n_splits=5)
linear_cv = cross_validate(
    linear_model, X, y, groups=groups, cv=cv,
    scoring={'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
    n_jobs=-1
)
print(f'Mean CV MAE: {-linear_cv["test_mae"].mean():.4f}')
print(f'Mean CV R2: {linear_cv["test_r2"].mean():.6f}')


##### Which hyperparameter optimization technique have you used and why?

GroupKFold cross-validation was used rather than hyperparameter tuning because ordinary Linear Regression has no meaningful regularization hyperparameters. The purpose is to validate that baseline performance is stable across journey groups.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Cross-validation provides a more reliable estimate of baseline generalization, but the linear model remains limited by its additive functional form. The major improvement comes from moving to non-linear tree models.


### ML Model - 2

In [ ]:
# ML Model - 2: Decision Tree Regressor
decision_tree = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', DecisionTreeRegressor(random_state=RANDOM_STATE, min_samples_leaf=2))
])
decision_tree.fit(X_train, y_train)
dt_pred = decision_tree.predict(X_test)
dt_results = regression_metrics(y_test, dt_pred, 'Decision Tree')
display(pd.DataFrame([dt_results]))


Decision Tree Regression captures non-linear thresholds and interactions without requiring explicit interaction terms. In the validated benchmark it achieved approximately **MAE 0.0044, RMSE 0.5298 and R² 0.999998**. The near-perfect score is explained by the dataset’s fixed route/class/agency fare structure.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Decision Tree hyperparameter tuning with grouped cross-validation
param_grid = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_leaf': [1, 2, 5]
}

dt_grid = GridSearchCV(
    estimator=Pipeline([
        ('preprocessor', tree_preprocessor),
        ('model', DecisionTreeRegressor(random_state=RANDOM_STATE))
    ]),
    param_grid=param_grid,
    cv=GroupKFold(n_splits=3),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
dt_grid.fit(X_train, y_train, groups=groups_train)
print('Best parameters:', dt_grid.best_params_)
print('Best grouped CV RMSE:', -dt_grid.best_score_)


##### Which hyperparameter optimization technique have you used and why?

GridSearchCV with GroupKFold was used because the parameter space is small and interpretable. Grouped folds preserve the journey structure during validation.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Tuning is expected to confirm that shallow restrictions can slightly reduce overfitting, but the dataset already produces extremely low error with a standard tree. The Random Forest is still preferred for ensemble stability.


#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

MAE represents the average absolute pricing error in the same currency units as the target. RMSE penalizes larger pricing misses more strongly, which matters when expensive flights are mispriced. R² measures the share of price variance explained by the model. For deployment, MAE and RMSE are most actionable because they translate directly into expected prediction error, while R² provides an overall goodness-of-fit summary.


### ML Model - 3

In [ ]:
# ML Model - 3: Random Forest Regressor
random_forest = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])
random_forest.fit(X_train, y_train)
rf_pred = random_forest.predict(X_test)
rf_results = regression_metrics(y_test, rf_pred, 'Random Forest')

results_df = pd.DataFrame([linear_results, dt_results, rf_results]).sort_values('RMSE')
display(results_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x='Model', y='RMSE', color='C0')
plt.title('Model Comparison by RMSE')
plt.xticks(rotation=15)
plt.show()


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

Random Forest averages multiple decision trees, reducing the variance of a single tree while preserving the ability to model non-linear interactions. In the validated benchmark with 20 trees it achieved approximately **MAE 0.0047, RMSE 0.1562 and R² 0.9999998**, outperforming the single tree on RMSE.


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Random Forest tuning with a compact grouped grid
rf_param_grid = {
    'model__n_estimators': [20, 50],
    'model__max_depth': [None, 20],
    'model__min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(
    estimator=Pipeline([
        ('preprocessor', tree_preprocessor),
        ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    param_grid=rf_param_grid,
    cv=GroupKFold(n_splits=3),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

# A reproducible sample keeps Colab tuning time practical while preserving group labels.
rng = np.random.default_rng(RANDOM_STATE)
sample_size = min(80000, len(X_train))
sample_positions = rng.choice(len(X_train), size=sample_size, replace=False)
rf_grid.fit(
    X_train.iloc[sample_positions],
    y_train.iloc[sample_positions],
    groups=groups_train.iloc[sample_positions]
)
print('Best parameters:', rf_grid.best_params_)
print('Best grouped CV RMSE:', -rf_grid.best_score_)


##### Which hyperparameter optimization technique have you used and why?

A compact GridSearchCV was used with GroupKFold. The search is run on a reproducible training sample to keep Colab execution practical while still preserving grouped validation. The final production fit can then use the selected parameters on the full training set.


##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The Random Forest reduces the already small single-tree error and gives the lowest RMSE among the tested models. Because the underlying dataset has deterministic fare combinations, the practical improvement is numerically small but consistent.


### 1. Which Evaluation metrics did you consider for a positive business impact and why?

The primary selection metrics are **RMSE and MAE**. RMSE is emphasized because large pricing errors are especially undesirable for a customer-facing travel application, while MAE remains directly interpretable as the typical absolute prediction error. R² is included as a complementary summary metric.


### 2. Which ML model did you choose from the above created models as your final prediction model and why?

The **Random Forest Regressor** is selected as the final flight-price model. It achieved the lowest held-out RMSE in the validated benchmark, captures the non-linear interaction between route, flight type and agency, and is more stable than a single decision tree. The model is saved together with its preprocessing pipeline so deployment uses exactly the same transformations as training.


### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Feature importance for the final Random Forest model
fitted_preprocessor = random_forest.named_steps['preprocessor']
fitted_model = random_forest.named_steps['model']

feature_names = categorical_features + numerical_features
importance_df = (
    pd.DataFrame({
        'feature': feature_names,
        'importance': fitted_model.feature_importances_
    })
    .sort_values('importance', ascending=False)
)

display(importance_df)
plt.figure(figsize=(9, 5))
sns.barplot(data=importance_df, x='importance', y='feature', color='C0')
plt.title('Random Forest Feature Importance')
plt.show()


## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the best-performing model for deployment
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'flight_price_model.joblib'

joblib.dump(random_forest, MODEL_PATH, compress=3)
print(f'Model saved to: {MODEL_PATH.resolve()}')


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Reload the model and perform an unseen-input sanity check
loaded_model = joblib.load(MODEL_PATH)

unseen_example = pd.DataFrame([{
    'from': X_test.iloc[0]['from'],
    'to': X_test.iloc[0]['to'],
    'flightType': X_test.iloc[0]['flightType'],
    'agency': X_test.iloc[0]['agency'],
    'time': X_test.iloc[0]['time'],
    'distance': X_test.iloc[0]['distance'],
    'year': int(X_test.iloc[0]['year']),
    'month': int(X_test.iloc[0]['month']),
    'dayofweek': int(X_test.iloc[0]['dayofweek'])
}])

prediction = loaded_model.predict(unseen_example)[0]
print('Input:')
display(unseen_example)
print(f'Predicted flight price: {prediction:.2f}')


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

The flight-price regression workflow successfully transformed a clean travel dataset into a deployment-ready machine-learning pipeline. Exploratory analysis showed that flight prices are strongly related to route distance and time, but categorical interactions involving origin, destination, flight type and agency are especially important. A leakage-aware grouped train/test split was used so paired records from the same travel journey could not cross between training and testing data.

Linear Regression provided a strong baseline but was limited by its additive structure. Decision Tree Regression captured the non-linear pricing rules and achieved near-perfect accuracy. Random Forest Regression produced the lowest held-out RMSE and was selected as the final model. The exceptionally high tree-based performance must be interpreted in the context of the supplied dataset: each route, flight type and agency combination effectively has a fixed price. Therefore, the model is highly reliable for combinations represented by the dataset but should be monitored carefully if deployed on new routes, agencies or future dynamic pricing data.

The final preprocessing and Random Forest estimator are stored together in one Joblib pipeline. This makes inference reproducible and prepares the model for the next stages of the Voyage Analytics capstone: Flask REST API serving, Docker containerization, Kubernetes scaling, Airflow scheduling, Jenkins CI/CD and MLflow experiment tracking.


### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***